#### 图像分类
- 最近邻
- 线性分类

计算机如何认识图像？

图像在计算机中表示为多维张量，具有 RGB 三个通道，每个通道的数值范围在 [0, 255] 之间。

传统方法通过边缘检测器寻找角点等特征，但这类方法泛化能力有限。

最近邻效果并不理想  
像素级比较对微小位移、光照变化非常敏感，且测试阶段计算量极大。

数据驱动方法  
1. 收集数据集和标签  
2. 用机器学习算法训练分类器  
3. 在新图像上测试分类器  

其中，训练集通常表示为：

$$
\{(x_i, y_i)\}_{i=1}^N
$$

其中 $x_i$ 表示第 $i$ 张输入图像，$y_i$ 表示其对应的真实标签，$N$ 为样本总数。

模型的目标是学习一个映射函数：

$$
f: X \to Y
$$

使得该函数能够将输入图像 $x$ 正确映射到其类别标签 $y$。

In [ ]:
import numpy as np
from typing import Any

def train(images: np.ndarray, labels: np.ndarray) -> Any:
    """
    Train a machine learning model.
    images: training images, shape (N, D)
    labels: training labels, shape (N,)
    Returns the trained model.
    """
    # Machine learning model training logic
    pass

def predict(model: Any, test_images: np.ndarray) -> np.ndarray:
    """
    Use the trained model to predict labels for test_images.
    model: trained model
    test_images: test images, shape (M, D)
    Returns predicted labels, shape (M,)
    """
    # Prediction logic
    pass

##### 最近邻分类器Nearest Neighbor

Distance Magic

距离计算是最近邻分类器的核心，用于衡量图片像素之间的相似度。

**L1距离**

L1距离又称曼哈顿距离，其几何特性是只能沿着坐标轴横着或竖着移动。

$$
d_1(I_1, I_2) = \sum_p |I_1^p - I_2^p|
$$

其中 $I_1$ 和 $I_2$ 代表两张图像，$p $为像素索引。

L1距离与L2距离的直观对比如下图所示：

![L1与L2距离对比图](../img/L1L2.png)

In [ ]:
import numpy as np
''' 
一个基于L1距离的最近邻图像分类器
它在训练阶段只是“死记硬背”所有图片和标签
在预测阶段则计算新图片与所有训练图片的像素差
找到最相似的那张图，直接把它的标签作为预测结果
'''

'''
输入：

X（图片数据）：是一个二维矩阵（Numpy Array），形状为 (N, D)。

N：图片的数量。比如训练集有50000张图。

D：每张图展平后的像素总数。32×32×3（RGB三通道） = 3072。

所以 X 就是一个 (50000, 3072) 的大矩阵，每一行代表一张被拉直成一条线的图片。

y（真实标签）：是一个一维数组，形状为 (N,)。

比如 y = [0, 5, 3, 2, ...]，里面的数字是 0 到 9，对应 10 个类别。

测试集输入：predict 里的 X 形状是 (M, D)，M 是测试图片数量（比如10000），D 依然是 3072。

'''

'''
输出：

X[i, :] 取出了第 i 张测试图片（一个长度为 D 的一维向量）。

self.Xtr - X[i, :] 利用了 NumPy 的广播机制，让所有训练图片（50000行）都减去这张测试图片。

np.abs(...) 取绝对值。

np.sum(..., axis=1) 把每行加起来，得到 50000 个距离值（L1距离）。

distances 是一个长度为 50000 的数组，代表这张测试图与所有训练图的像素差之和。

找出 distances 中最小值的索引，也就是“最像”的那张训练图的下标。

把最像的那张训练图的标签，直接作为第 i 张测试图的预测结果。

'''
class NearestNeighbor:
    def __init__(self):
        pass

    def train(self, X, y):
        # X is N x D where each row is an example.
        # Y is 1-dimensional of size N.
        # The nearest neighbor classifier simply remembers all the training data.
        self.Xtr = X
        self.ytr = y

    def predict(self, X):
        # X is N x D where each row is an example we wish to predict label for.
        num_test = X.shape[0]

        # Let's make sure that the output type matches the input type.
        Ypred = np.zeros(num_test, dtype=self.ytr.dtype)

        # Loop over all test rows.
        for i in range(num_test):
            # Find the nearest training image to the i'th test image
            # using the L1 distance (sum of absolute value differences).
            distances = np.sum(np.abs(self.Xtr - X[i, :]), axis=1)

            min_index = np.argmin(distances)  # Get the index with smallest distance.
            Ypred[i] = self.ytr[min_index]    # Predict the label of the nearest example.

        return Ypred


**L2距离**

L2距离又称欧氏距离，度量的是两点之间的直线距离。

数学公式如下：

$$
d_2(I_1, I_2) = \sqrt{\sum_p (I_1^p - I_2^p)^2}
$$

其中 $I_1$ 和 $I_2$ 代表两张图像，$p$ 为像素索引。

L1与L2的对比可参考前文的图示。

---

##### K-Nearest Neighbors

K近邻分类器是最近邻的推广。K=1时即为最近邻分类器，预测时取最近的1个训练样本的标签作为结果。K>1时，则选取距离最近的K个训练样本，通过投票决定最终预测类别。

[K-Nearst Neighbors 演示](http:vision.stanford.edu/teaching/cs231n-demos/knn/)

##### 超参数

K近邻算法中的K值和距离函数都是典型的超参数。超参数需要人为设置，不能由算法从数据中自动学习得到。

**K值的选择**

K值过小，模型对噪声敏感，容易过拟合。K=1时在训练集上永远能达到100%准确率，但这不代表模型泛化能力强。

K值过大，模型过于平滑，可能欠拟合。

**超参数选取策略**

常见的超参数选取策略有以下几种。

Idea #1：选择在训练集上表现最好的超参数。这是错误的，因为K=1永远在训练集上完美。

Idea #2：选择在测试集上表现最好的超参数。这也是错误的，这样会导致算法对测试集过拟合，实际部署时性能会远低于预期。

Idea #3：将数据分为训练集、验证集，在验证集上选择超参数，最后在测试集上评估。这是正确的做法。

Idea #4：交叉验证。将训练集分成若干份，轮流将其中一份作为验证集，其余作为训练集，最后取平均结果。适用于数据集较小的情况。

---

##### 数据集划分

**3. 训练集、验证集、测试集**

将可用数据划分为三部分：

- **训练集**：用于训练模型参数。
- **验证集**：用于调整超参数，选择最优配置。
- **测试集**：只在最后使用一次，用于评估最终模型的泛化性能。

以CIFAR-10为例，可以用49000张作为训练集，1000张作为验证集。

**Idea #3的具体流程**

在验证集上尝试不同的超参数，记录每个超参数对应的准确率，选择验证集上表现最好的超参数。然后用这个超参数在全部训练数据上重新训练，最后在测试集上跑一次，报告结果。

**4. 交叉验证**

当训练数据较少时，验证集数量也会很少，此时可以使用交叉验证。

将训练集平均分成 $k$ 份（通常k=3、5、10），每次用其中 $k-1$ 份训练，剩下1份验证，循环 $k$ 次，最后取 $k$ 次验证结果的平均值作为该超参数的性能估计。

这样做的好处是减少了验证集划分带来的噪声，得到更稳定的超参数选择。缺点是计算成本成倍增加。

如果训练数据充足，通常优先使用单次验证集划分，因为交叉验证计算开销较大。

##### 距离度量的局限性

K近邻使用像素级距离进行图像分类，实际效果并不理想。

原因有二：

第一，像素距离对微小的位移、旋转、光照变化非常敏感。同一物体经过平移或旋转后，像素值差异可能很大，导致被判定为不同类别。

第二，高维空间中的距离度量会失去区分度，这种现象被称为维数灾难。随着维度增加，所有点之间的距离趋于接近，距离度量不再具有信息量。

因此，K近邻搭配像素距离在图像分类中几乎不被实际使用。

---

##### 线性分类器 Linear Classifier

**映射**

线性分类器是一个把输入映射到输出的函数 $f(x, W)$，其中 $W$ 是权重矩阵。

输入图像大小为 $32 \times 32 \times 3$，展平后得到长度为 3072 的向量 $x$。函数 $f(x, W)$ 将其映射为 10 个类别的分数。

线性模型公式如下：

$$
f(x, W) = W x + b
$$

一张图像会对每个类别都输出一个分数。

线性分类器是神经网络的基础模块。

![f(x,W)](../img/f(x,W).png)

**损失函数与最大似然估计**

损失函数用于衡量预测分数与真实分数之间的差异。

基于最大似然估计，计算正确类别的概率，取对数，再取负值，就得到了损失。

**Softmax 公式**

Softmax 分类器把原始分数转换为概率分布。它本质上就是多分类逻辑回归。

Softmax 函数将分数 $s$ 映射为合为1概率：

$$
P(y = k | x) = \frac{e^{s_k}}{\sum_j e^{s_j}}
$$

负值概率趋于0。

对应的损失函数即交叉熵损失：

$$
L_i = -\log P(y_i | x_i)
$$

它衡量的是模型预测分布与真实分布之间的差异。


---

**图像识别时计算机会遇到很多挑战。**

![challenges](../img/COR.png)

#### 正则化和优化


##### 正则化 Regularization

核心思路：在训练集上表现稍差，但在未见过的数据上表现更好。倾向于选择拟合度稍低但更简单的模型。

通常不对偏置项进行正则化，因为偏置不控制特征的方向。

完整的损失函数由数据损失和正则化项组成：

$$
L = \frac{1}{N} \sum_i L_i + \lambda R(W)
$$

其中 $\lambda$ 是正则化强度，$R(W)$ 是正则化项。

**L2正则化**

对权重平方进行惩罚。对极小值的惩罚更小，倾向于让权重分散。

$$
R(W) = \sum_k \sum_l W_{k,l}^2
$$

**L1正则化**

对权重绝对值进行惩罚。倾向于产生稀疏的权重矩阵。

$$
R(W) = \sum_k \sum_l |W_{k,l}|
$$

**L1 + L2**

结合两者，又称 Elastic Net。

![Regularization](../img/Regularization.png)

##### 优化 Optimization

优化的目标是找到损失函数的最低点，即最优解。

核心方法是梯度下降。沿着梯度的反方向向下走，感受当前位置的损失，然后向下迈一步。Follow the slope。

**数值梯度与解析梯度**

数值梯度利用极限定义，取极小的 $h$ 来近似计算梯度。计算慢，但容易实现，常用于梯度检查。

$$
\frac{df}{dx} \approx \frac{f(x+h) - f(x-h)}{2h}
$$

解析梯度通过微积分推导得出，计算精确且快速，是反向传播使用的真正方法。

梯度检查用于验证解析梯度是否实现正确。

损失函数通常是可微的。对于凸函数，局部最小值就是全局最小值。

**梯度下降**

定好迭代次数，或者等待损失收敛。

##### 随机梯度下降 SGD

每次迭代只使用一小批数据来估计梯度。

SGD的问题在于：

1. 鞍点。梯度为0，容易卡住。
2. 噪声。子采样带来的梯度估计噪声，导致更新方向震荡。

##### SGD + Momentum

引入动量，对噪声进行平均，抑制震荡。

更新公式为：

$$
v = \rho v - \alpha \nabla L
$$

$$
x = x + v
$$

其中 $\rho$ 是动量系数，通常取 0.9 左右。$\alpha$ 是学习率。

动量能让收敛可能更慢，但容易找到更优的极小值。因为积累了历史速度，可能会在最小值附近超调，但通过后续调整最终能稳定下来。

In [ ]:
# gradient descent

def compute_gradient(x ,batch_data):
    # Compute the gradient of the loss function with respect to x
    # This is a placeholder function; replace with actual gradient computation
    return np.random.randn(*x.shape)  # Example: random gradient for demonstration


# SGD + Momentum

vx = 0  # Initialize velocity as zero vector
while True:
    dx = compute_gradient(x, batch_data)
    # Placeholder: returns a random array with the same shape as x
    # In practice, this should be the real gradient (e.g., 2*x for f(x)=x^2)

    vx = rho * vx + dx  # v = rho * v + gradient
    x -= learning_rate * vx



##### RMSProp

RMSProp 的核心思路是：**少往陡峭的地方走，多往平坦的地方走。**

它通过计算梯度平方的指数衰减平均，来自适应地调整每个参数的学习率。

更新公式为：

$$
s = \beta s + (1 - \beta) (\nabla L)^2
$$

$$
x = x - \alpha \frac{\nabla L}{\sqrt{s} + \epsilon}
$$

其中 $\beta$ 是衰减率，通常取 0.9 或 0.99。

当梯度大（陡峭）时，$s$ 大，分母大，实际步长变小。当梯度小（平坦）时，$s$ 小，分母小，实际步长变大。

##### Adam (almost)

Adam 本质上是 RMSProp 和 Momentum 的结合。

它同时计算梯度的一阶矩估计（动量）和二阶矩估计（梯度平方），并引入**偏差校正**。

偏差校正解决了初始步长过大的问题。因为在训练初期，$m$ 和 $v$ 初始化接近 0，如果不校正，更新量会非常小。校正后，早期更新步长被放大，模型能快速启动。

Adam 中动量计算时的梯度只看数据损失，最后加上正则化。完整的梯度为：

$$
g_t = \nabla f(w_t) + \lambda w_t
$$

其中 $\lambda w_t$ 是正则化项。

更新公式为：

$$
m = \beta_1 m + (1 - \beta_1) g_t
$$

$$
v = \beta_2 v + (1 - \beta_2) g_t^2
$$

$$
\hat{m} = \frac{m}{1 - \beta_1^t}, \quad \hat{v} = \frac{v}{1 - \beta_2^t}
$$

$$
x = x - \alpha \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}
$$

##### AdamW

标准的 Adam 在处理正则化时存在缺陷。因为 L2 正则化被加进了梯度里，它会和自适应学习率交互，导致正则化效果被削弱。

AdamW 的核心是**解耦权重衰减（Decoupled Weight Decay）**。它将权重衰减从梯度更新中分离出来，直接作用于权重本身。

更新公式为：

$$
x = x - \alpha \left( \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon} + \lambda x \right)
$$

**关于“跑固定轮次后，学习率除以10”**：这不是 AdamW 的专利，而是**步长衰减（StepLR）**。这是一种通用的学习率调度策略，通常每跑固定轮次后，学习率乘以 0.1。它和 AdamW 是正交的概念，可以搭配使用。

##### 余弦学习率衰减

学习率遵循余弦曲线，从初始值平滑衰减到 0 或一个小值。

$$
\alpha_t = \frac{1}{2} \alpha_0 \left( 1 + \cos\left( \frac{t \pi}{T} \right) \right)
$$

其中 $T$ 是总轮次。相比 StepLR，余弦衰减更平滑，后期学习率极小，有助于模型精细收敛。

##### 线性预热

在训练刚开始时，模型权重是随机的，梯度可能很大。如果直接用大学习率，容易导致训练不稳定。

线性预热在最初的几个轮次里，将学习率从 0 或一个极小值线性增加到初始学习率。

```python
# Linear Warmup (pseudocode)
if current_step < warmup_steps:
    lr = base_lr * current_step / warmup_steps
else:
    lr = base_lr
```

##### 线性缩放定律

这是一个经验法则。当批量大小乘以 $k$ 时，学习率也应该乘以 $k$（或 $\sqrt{k}$）。这有助于在大批量训练时保持梯度更新的方差大致恒定。

##### 海森矩阵与大模型

海森矩阵是二阶导数矩阵，能提供损失函数的曲率信息，理论上能帮助优化器找到更好的下降方向。

但在大模型中基本不用。因为参数量巨大，海森矩阵的维度是平方级别，计算和存储成本极高。即使是对角近似，也极其昂贵。因此，大模型训练几乎全部依赖 Adam、AdamW 等一阶优化方法。

In [ ]:
import numpy as np

# Adam Optimizer

first_moment = 0        # m: first moment estimate 
#first_moment = np.zeros_like(x)

second_moment = 0       # v: second moment estimate

beta1 = 0.9             # Decay rate for first moment
beta2 = 0.999           # Decay rate for second moment
learning_rate = 0.001
epsilon = 1e-8

for t in range(1, num_iterations + 1):
    dx = compute_gradient(x)    # g_t: gradient at current step
    
    # Update biased first moment estimate (m = beta1 * m + (1 - beta1) * g)
    first_moment = beta1 * first_moment + (1 - beta1) * dx
    
    # Update biased second raw moment estimate (v = beta2 * v + (1 - beta2) * g^2)
    second_moment = beta2 * second_moment + (1 - beta2) * (dx ** 2)
    
    # Compute bias-corrected first moment estimate
    first_moment_corrected = first_moment / (1 - beta1 ** t)
    
    # Compute bias-corrected second raw moment estimate
    second_moment_corrected = second_moment / (1 - beta2 ** t)
    
    # Update parameters (x = x - lr * m_hat / (sqrt(v_hat) + epsilon))
    x -= learning_rate * first_moment_corrected / (np.sqrt(second_moment_corrected) + epsilon)


In [ ]:
# AdamW (Decoupled Weight Decay) 解耦权重衰减,修改最后一行
x -= learning_rate * (first_moment_corrected / (np.sqrt(second_moment_corrected) + epsilon) + weight_decay * x)

##### SVM损失函数

**用途**

SVM损失是数据损失的核心组成部分，用于衡量模型输出的预测分数与真实标签之间的差距。它通常用于线性分类器或神经网络的输出层，为模型提供优化方向。

**核心原理：安全边界与合页损失**

SVM的核心思想是：不仅要预测正确，还要自信地预测正确。它希望正确类别的分数，比其他所有错误类别的分数，至少高出一个安全边界，即 Margin，通常设为 $\Delta = 1.0$。

如果正确类别的分数比某个错误类别的分数高出至少 $\Delta$，模型就认为在这个类别上已经足够安全，不再产生损失，即 Loss = 0。否则，就会产生惩罚，即 Loss > 0。这种机制被称为合页损失 Hinge Loss。

**数学公式**

对于第 $i$ 个样本，多类SVM损失的公式为：

$$
L_i = \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + \Delta)
$$

其中 $s_j$ 是模型对第 $j$ 个错误类别的预测分数，$s_{y_i}$ 是模型对真实类别 $y_i$ 的预测分数，$\Delta$ 是安全边界，通常取 1.0。$\max(0, \cdot)$ 就是合页损失，如果括号内小于0，即已经足够安全，则损失为0。

整个数据集的平均SVM损失为：

$$
L = \frac{1}{N} \sum_{i=1}^N L_i + \lambda R(W)
$$

其中 $\lambda R(W)$ 是正则化项。

**直观例子**

假设有3个类别，分别是猫、狗、汽车，真实标签是猫，即 $y_i = 0$。模型的预测分数为：猫 3.2，狗 5.1，汽车 -1.7。设边界 $\Delta = 1.0$。

对于错误类别狗，$\max(0, 5.1 - 3.2 + 1.0) = \max(0, 2.9) = 2.9$，产生损失。

对于错误类别汽车，$\max(0, -1.7 - 3.2 + 1.0) = \max(0, -3.9) = 0$，不产生损失。

所以这个样本的损失为 2.9 + 0 = 2.9。

这说明模型虽然预测对了猫，但只比狗高了1.9分，没有达到安全边界1.0的自信差距，因此被罚款。

**SVM损失的作用**

指导优化方向。它为模型提供了一个可微的梯度信号，驱动模型去拉大正确类别与错误类别之间的分数差距。

控制模型行为。边界 $\Delta$ 决定了模型要多自信才算满意。$\Delta$ 越大，模型被迫拉开的分数差距就越大。

与Softmax的对比。SVM只关心分数是否超过边界，不关心分数之间的绝对差异。而Softmax，也就是交叉熵，则会把分数转化为概率分布，关心正确类别的概率有多大。这是两种不同的优化哲学。

---

#### 神经网络与反向传播

##### SVM损失函数
SVM损失函数通常用于线性分类器，用来衡量预测分数与真实分数之间的差异。它属于数据损失的一部分。

##### 线性映射
线性分类器的基础公式是：

$$
f = Wx
$$

这种模型只能解决线性可分的问题。

##### 两层神经网络
在线性模型的基础上引入隐藏层，得到两层神经网络：

$$
f = W_2 \max(0, W_1 x)
$$

其中 $W_1$ 是第一层权重，$W_2$ 是第二层权重。$\max(0, \cdot)$ 即 ReLU 激活函数。完整形式通常会加上偏置项 $b_1$ 和 $b_2$：

$$
f = W_2 \max(0, W_1 x + b_1) + b_2
$$

##### 激活函数
激活函数引入非线性机制。如果没有激活函数，多层神经网络无论叠多深，本质上仍然等价于一个线性变换。

**ReLU**

ReLU 即整流线性单元。公式为：

$$
f(x) = \max(0, x)
$$

计算简单，收敛速度快，是当前最常用的默认激活函数。

**死神经元**

当某个神经元的权重使得其对所有输入都输出负数时，ReLU 的梯度为 0。该神经元将永久失活，不再更新，这叫死神经元问题。

**Leaky ReLU**

为了解决死神经元问题，Leaky ReLU 在负半轴引入一个小斜率 $\alpha$：

$$
f(x) = \max(\alpha x, x)
$$

通常 $\alpha$ 取 0.01 左右。

**ELU**

ELU 即指数线性单元，在负半轴使用指数函数：

$$
f(x) = \begin{cases} x & x > 0 \\ \alpha(e^x - 1) & x \le 0 \end{cases}
$$

负半轴均值接近 0，有助于加速收敛，但计算量稍大。

**etc...**

还有其他变体，例如 GELU、Swish、Maxout 等。

##### 制造非线性
激活函数的作用就是制造非线性。没有非线性，再深的网络也只是线性模型。

##### 全连接神经网络
全连接神经网络也叫多层感知机 MLP。每一层的每个神经元都与前一层的所有神经元相连。通过堆叠多个全连接层并配合激活函数，网络可以拟合极其复杂的非线性函数。

例如，在一个简单的线性分类器中，前向传播可能仅仅是：

$$
f = W x + b
$$

而在一个简单的两层神经网络中，前向传播引入了激活函数，可能如下：

$$
f = W_2 \max(0, W_1 x + b_1) + b_2
$$



In [ ]:
# forward pass 
import numpy as np

def forward_pass(x, W1, b1, W2, b2):
    """
    Simple forward pass for a 2-layer neural network.
    x: input data, shape (N, D)
    W1, b1: weights and bias of the first layer
    W2, b2: weights and bias of the second layer
    Returns the output scores.
    """
    # First layer: linear transformation followed by ReLU activation
    hidden = np.maximum(0, x @ W1 + b1)
    
    # Second layer: linear transformation to output scores
    scores = hidden @ W2 + b2
    
    return scores

In [ ]:
# a 2-layer neural network 

import numpy as np

# Define network dimensions
N, D_in, H, D_out = 64, 1000, 100, 10

# Randomly initialize input data, target, and weights
x = np.random.randn(N, D_in)
y = np.random.randn(N, D_out)
w1 = np.random.randn(D_in, H)
w2 = np.random.randn(H, D_out)

learning_rate = 1e-4

for t in range(500):
    # Forward pass: compute predicted y
    # First layer: linear transformation followed by Sigmoid activation
    h = 1 / (1 + np.exp(-x.dot(w1)))
    # Second layer: linear transformation to output
    y_pred = h.dot(w2)

    # Compute loss using squared error
    loss = np.square(y_pred - y).sum()
    if t % 100 == 0:
        print(f"Iteration {t}, Loss: {loss}")

    # Backward pass: compute gradients
    # Gradient of loss with respect to y_pred
    grad_y_pred = 2.0 * (y_pred - y)
    # Gradient of loss with respect to w2
    grad_w2 = h.T.dot(grad_y_pred)
    # Gradient of loss with respect to hidden layer output h
    grad_h = grad_y_pred.dot(w2.T)
    # Add Sigmoid derivative: multiply by h * (1 - h)
    grad_h = grad_h * h * (1 - h)
    # Gradient of loss with respect to w1
    grad_w1 = x.T.dot(grad_h)

    # Update weights using gradient descent
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2


##### 正则化与网络规模的区别

**正则化**用于限制模型权重的大小，防止过拟合，比如 **L1 和 L2 正则化**。**网络规模**指层数和神经元数量，属于模型架构设计。不能用**缩小网络规模**来替代正则化。缩小网络会直接降低模型的表示能力，容易导致**欠拟合**，而正则化是在保持模型容量的前提下约束参数。

##### 神经元与激活函数

神经网络中的每个神经元可以视为计算图中的一个**门单元**。**激活函数**是一类特殊的门，引入**非线性**。前向传播时，它将输入映射为输出。反向传播时，它根据输入和输出计算**局部梯度**，并将**上游梯度**传递给**下游梯度**。

##### 计算图

![计算图](../img/backpropagation.png)

**计算图**将复杂函数拆解为一系列简单的中间步骤。**前向传播**时，依次计算中间节点的输出并保存。**反向传播**时，利用**链式法则**从输出层向输入层逐级求导。

##### 反向传播的目标与流程

反向传播的目的是计算**损失 $L$** 对所有变量包括**权重 $W$** 和**偏置 $b$** 的梯度。这些梯度将被**优化器**用来更新权重。

在实际框架中，不会为每一层手动写出导数函数，而是通过逐级反向传播自动完成梯度计算。

反向传播的完整流程如下。

1. **前向传播**求出每一步的中间输入和输出，保存在内存中供后续求导使用。
2. **反向传播**从末端开始。**末端梯度**通常恒为 1，即 $\frac{\partial L}{\partial L} = 1$。
3. 每一步先求**局部梯度**，再乘以**上游梯度**，得到**下游梯度**，并将其继续向下游传递。

##### 门单元及其梯度传播规则

**加法门**是**梯度分配器**。对于 $z = x + y$，局部梯度 $\frac{\partial z}{\partial x} = 1$，$\frac{\partial z}{\partial y} = 1$。加法门将上游梯度原封不动地分发给两个输入。

**乘法门**是**梯度交换器**。对于 $z = x \cdot y$，局部梯度 $\frac{\partial z}{\partial x} = y$，$\frac{\partial z}{\partial y} = x$。乘法门把上游梯度乘以另一个输入的值后再分发。

**复制门**用于把一个变量复制到多个分支。反向传播时，来自不同分支的梯度会在复制点进行累加，因为同一个变量对多个输出都有贡献。

**最大值门**是**梯度路由器**。对于 $z = \max(x, y)$，梯度只会传递给前向传播中数值较大的那个输入，另一个输入的梯度为 0。

##### Sigmoid 门

**Sigmoid 函数**作为激活函数时，其局部梯度为 $\sigma'(z) = \sigma(z)(1 - \sigma(z))$。反向传播时，上游梯度乘以这个局部梯度得到下游梯度。由于 Sigmoid 的导数最大值仅为 0.25，在深层网络中使用会导致梯度逐层衰减，产生**梯度消失问题**。


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 前向传播
def forward_pass(w0, x0, w1, x1, w2):
    s0 = w0 * x0        # 乘法门
    s1 = w1 * x1        # 乘法门
    s2 = s0 + s1        # 加法门
    s3 = s2 + w2        # 加法门
    L = sigmoid(s3)     # Sigmoid 门
    return s0, s1, s2, s3, L

# 前向传播计算
w0, x0, w1, x1, w2 = 2.0, 3.0, 1.0, 4.0, 0.5
s0, s1, s2, s3, L = forward_pass(w0, x0, w1, x1, w2)

# 反向传播
grad_L = 1.0            # 末端梯度恒为 1，即 dL/dL
grad_s3 = grad_L * (L * (1 - L))    # Sigmoid 门：乘以局部梯度 L*(1-L)

grad_s2 = grad_s3 * 1.0             # 加法门：梯度原样分发
grad_w2 = grad_s3 * 1.0             # 加法门：梯度原样分发

grad_s0 = grad_s2 * 1.0             # 加法门：梯度原样分发
grad_s1 = grad_s2 * 1.0             # 加法门：梯度原样分发

grad_w0 = grad_s0 * x0              # 乘法门：乘以另一个输入 x0
grad_x0 = grad_s0 * w0              # 乘法门：乘以另一个输入 w0

grad_w1 = grad_s1 * x1              # 乘法门：乘以另一个输入 x1
grad_x1 = grad_s1 * w1              # 乘法门：乘以另一个输入 w1

print(f"L: {L}")
print(f"grad_w0: {grad_w0}, grad_x0: {grad_x0}")
print(f"grad_w1: {grad_w1}, grad_x1: {grad_x1}")
print(f"grad_w2: {grad_w2}")

In [ ]:
import torch

# 前向传播和反向传播的接口
class Multiply(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input1, input2):
        # ctx 是上下文对象，用于在前向和反向传播之间共享数据
        # 保存输入张量，反向传播时需要用它们计算局部梯度
        ctx.save_for_backward(input1, input2)
        # 前向计算：返回两个输入的乘积
        return input1 * input2

    @staticmethod
    def backward(ctx, grad_output):
        # grad_output 是损失函数对 forward 输出结果的梯度（上游梯度）
        # 从上下文中取出前向传播时保存的输入张量
        input1, input2 = ctx.saved_tensors
        # 乘法门的局部梯度：对 input1 的梯度等于 grad_output 乘以 input2
        grad_input1 = grad_output * input2
        # 对 input2 的梯度等于 grad_output 乘以 input1
        grad_input2 = grad_output * input1
        # 返回损失对两个输入的梯度，顺序必须与 forward 的输入顺序一致
        return grad_input1, grad_input2



vector to vector

**向量对向量求导**

当函数输入是向量，输出也是向量时，对输出向量 $\mathbf{y}$ 的每个元素关于输入向量 $\mathbf{x}$ 的每个元素求偏导，得到雅可比矩阵。

$$
\mathbf{J} = \frac{\partial \mathbf{y}}{\partial \mathbf{x}} = 
\begin{bmatrix}
\frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{bmatrix}
$$

**损失函数 $L$ 是标量**

当最终损失 $L$ 是一个标量时，$L$ 对向量或矩阵求导的结果称为梯度，其形状与被求导的变量完全相同。比如 $L$ 对矩阵 $W$ 的梯度 $\frac{\partial L}{\partial W}$，其形状与 $W$ 一致。

**矩阵乘法的反向传播**

对于矩阵乘法 $y = xW$，输入 $x$ 是 $N \times D$ 的矩阵，权重 $W$ 是 $D \times M$ 的矩阵，输出 $y$ 是 $N \times M$ 的矩阵。

在反向传播时，不需要构造巨大的雅可比矩阵。$x$ 的第 $i$ 行只影响 $y$ 的第 $i$ 行。上游梯度 $\frac{\partial L}{\partial y}$ 的形状是 $N \times M$。根据链式法则，可以通过矩阵乘法直接求出对 $x$ 和 $W$ 的梯度：

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} W^T
$$

$$
\frac{\partial L}{\partial W} = x^T \frac{\partial L}{\partial y}
$$

为矩阵乘法写反向传播函数，利用矩阵乘法计算梯度，避免了逐个元素计算雅可比矩阵。

```python
import numpy as np

class MatMul:
    def __init__(self):
        self.x = None
        self.W = None

    def forward(self, x, W):
        # 保存前向传播的输入，反向传播计算局部梯度时需要用到
        self.x = x
        self.W = W
        # 前向计算：矩阵乘法
        return x.dot(W)

    def backward(self, dout):
        # dout 是上游传下来的梯度，形状为 (N, M)
        # 损失对输入 x 的梯度，形状为 (N, D)
        dx = dout.dot(self.W.T)
        # 损失对权重 W 的梯度，形状为 (D, M)
        dW = self.x.T.dot(dout)
        return dx, dW
```


**Recap**

用线性分类器解决图像分类，用张量定义输入输出，用权重矩阵 $W$ 预测损失得分，用损失函数判断 $W$ 的表现。线性分类并不强大，因此提出了神经网络，堆叠线性与非线性层。为了优化分类器，需要计算更复杂的 $W$，因此引入了计算图、梯度和反向传播。

机制上，可以定义各种节点，都遵循计算输出和局部梯度的接口。只要所有节点都遵循这个规则，就能组合成能进行任意计算的复杂大图。上游梯度的形状始终和输出完全一样，下游梯度是损失对输入的导数，形状和输入一样。反向传播的核心在于链式法则，通过将局部梯度逐级向上传递，自动求出损失对每一个参数的梯度，这为后续的梯度下降更新提供了依据。

##### 特征表示

传统方法是人工定义一种表示作为输入，比如颜色直方图、方向梯度直方图 HOG，但这类方法已被淘汰。这类人工特征提取需要大量专家知识，且泛化能力有限，因为它们丢失了图像的空间信息，无法捕捉复杂的纹理和语义。

现在的趋势是端到端设计，让数学和计算比人类更擅长寻找中间函数。网络通常由卷积层、池化层、非线性层和全连接层 MLP 组成。特征提取器不再是手工设计，而是完全由数据驱动，通过反向传播自动学习得到。这种方式使得网络可以针对具体任务自动提取最适合的特征。

##### 历史

LeNet-5 是最早的卷积神经网络之一，由 Yann LeCun 提出，用于手写数字识别。AlexNet 在 2012 年 ImageNet 竞赛中取得突破，引爆了深度学习，它引入了 ReLU 激活函数和 Dropout 来防止过拟合。随后 VGG 和 ResNet 通过更深的网络结构不断刷新性能，ResNet 更是引入了残差连接解决了深层网络梯度消失的问题。如今 Transformer 架构在部分任务中替代了卷积神经网络，比如 Vision Transformer 将图像分块后送入注意力机制处理。


---

#### 卷积网络

##### 卷积、滤波器与特征图的关系

这三者是卷积层最核心的组成，理解它们的物理关系至关重要。

**滤波器**是一个小的权重矩阵，例如 $3 \times 3$ 大小，它是提取特征的**工具**。滤波器内部的值就是网络需要学习的参数。

**卷积**是滤波器在图像上滑动的**动作**。滤波器按照步幅 $S$ 在图像上滑动，每到一个位置，就将滤波器内的权重与图像对应位置的$K \times K $个像素值进行逐元素相乘并求和，得到一个标量输出。这就是滤波器在该位置的匹配得分。这个计算本质上就是点积，也就是模板匹配。

**特征图**是卷积动作产生的**结果**。当滤波器扫完整张图，所有位置的得分排列成一个二维矩阵，这就是激活图或特征图。特征图上的每一个点，代表输入图像对应位置与滤波器模式的匹配程度。

为了提取多种不同的特征，通常会使用多个滤波器。每个滤波器生成一张特征图，多个滤波器并行工作，所有特征图汇成一个三维张量。单个滤波器的深度必须与输入的通道数 $C_{in}$ 保持一致。如果输入是 RGB 三通道，滤波器就变成 $3 \times 3 \times 3$ 的三维小块，逐通道计算点积后再把三个结果相加。

##### 卷积层的参数与计算

在卷积层中，区分参数和超参数很关键。卷积核数量、尺寸、步幅 $S$ 和填充 $P$ 都是超参数，是训练前定好的。滤波器内部的数值和偏置项是可学习参数，通过反向传播更新。反向传播时，同一个滤波器在不同空间位置的梯度会累加，因为参数共享。参数共享是卷积层高效的核心原因。

输出尺寸由以下公式决定：

$$
H_{out} = \left\lfloor \frac{H_{in} - K + 2P}{S} \right\rfloor + 1
$$

不加填充时，特征图会越来越小，边缘像素容易被忽略。引入 padding 补 0 可以保留边缘信息，保证网络可以不断加深。常见做法是 $K$ 取奇数，$P = (K - 1) / 2$。

卷积层通常批量处理四维张量。

- 输入维度为 $N, C_{in}, H, W$，

- 滤波器组维度为 $C_{out}, C_{in}, K_w, K_h$，

- 输出维度为 $N, C_{out}, H', W'$。

其中 $C_{out}$ 等于滤波器的数量。

在卷积层之间，数据的尺寸和通道变化是有严格规律的。当前层的输出通道数 $C_{out}$ 将直接作为下一层的输入通道数 $C_{in}$。例如，第一层输入通道为 3，设置 32 个滤波器，输出通道即为 32，那么第二层的输入通道必须相应为 32。通道数通常随网络加深而增加，空间尺寸则通过步幅大于 1 的卷积或池化层来主动减小，以扩大感受野并降低计算量。

卷积网络就是包含多个卷积层的计算图。卷积是点积，点积是线性算子，所以卷积层之间必须加激活函数引入非线性，比如ReLU。如果缺少非线性，无论叠多少层卷积，最终依然等价于一个单层线性变换。

在特征层次上，浅层滤波器倾向于学习颜色、边缘等低级特征，深层滤波器则组合这些低级特征，学习纹理和物体部件等高级语义特征。


##### 可学习参数数量

![CNN示例](../img/CNN_exa.png)

以图示为例，输入体积为 $3 \times 32 \times 32$，使用 10 个 $5 \times 5$ 滤波器，步幅 1，填充 2。输出体积为 $10 \times 32 \times 32$。

每个滤波器的参数数量为 $3 \times 5 \times 5 + 1 = 76$，其中 1 是偏置项。10 个滤波器的总参数数量为 $10 \times 76 = 760$。如果使用全连接层处理同样大小的输入输出，参数量将达到数千万级别。参数共享是卷积层高效的核心原因，一个滤波器在图像的不同位置共享相同的权重，极大地减少了参数量。

##### 感受野

![感受野](../img/Receptive_Fields.png)

有效感受野指原始图像中有多少像素能影响到网络后端的某个激活值。感受野越大，网络能看到的全局上下文信息就越多。感受野随层数线性扩大。假设第 $l$ 层的感受野为 $R_l$，卷积核大小为 $K_l$，步幅为 $S_l$，则下一层的感受野递推公式为：

$$
R_{l+1} = R_l + (K_{l+1} - 1) \times \prod_{i=1}^{l} S_i
$$

除了增加层数，还可以通过增大卷积核尺寸或使用步幅卷积来更快地扩大感受野。


##### 池化层

池化层是卷积网络中用于下采样的核心组件。卷积层负责提取特征，池化层负责对特征图进行压缩，通道数保持不变，但空间尺寸会显著减小。这能降低后续计算量，扩大有效感受野，并引入一定的平移不变性。池化层独立地在每个通道上操作。

核心思路是对图像的高宽维度做合并处理。

根据计算方式的不同，池化层主要分为以下几种类型。

**最大池化 Max Pooling**

这是最常用的池化方式。在每个池化窗口内，取出数值最大的那个元素，作为输出特征图对应位置的值。

举例来说，假设输入是一个 $4 \times 4$ 的单通道特征图，池化窗口大小为 $2 \times 2$，步长为 $2$。左上角 $2 \times 2$ 的窗口内如果有数值 $1, 3, 2, 5$，最大池化会输出 $5$。依次滑动窗口，$4 \times 4$ 的输入就被压缩成 $2 \times 2$ 的输出，空间尺寸减半。

最大池化保留了窗口内最强烈的激活值，相当于提取了局部最显著的特征，保留了纹理细节。在反向传播时，它只把上游梯度传给前向传播中数值最大的那个位置，其余位置的梯度为 0，这保证了梯度只通过最强激活的路径流动。

**平均池化 Average Pooling**

在每个池化窗口内，取所有元素的平均值，作为输出特征图对应位置的值。

以同样的 $2 \times 2$ 窗口为例，如果窗口内数值为 $1, 3, 2, 5$，平均池化会输出 $(1+3+2+5)/4 = 2.75$。

平均池化相当于对局部区域做了平滑，保留了整体背景信息。在反向传播时，它把上游梯度平均分配到窗口内的每一个位置，相当于每个输入元素都承担了均等的梯度贡献。平均池化是线性算子，而最大池化是非线性的。

**全局平均池化 Global Average Pooling**

这是一种特殊的平均池化。池化窗口的大小等于整个特征图的空间尺寸，直接把每个通道上的所有值求平均，输出一个数值。

全局平均池化常用于替代网络末端的全连接层。例如，输入特征图的维度是 $N, C, H, W$，经过全局平均池化后，输出维度变成 $N, C, 1, 1$。这能极大地减少参数量，防止过拟合，在现代网络如 ResNet 中被广泛使用。

**其他池化变体**

除了上述三种，还有一些变体用于特定场景。比如**随机池化 Stochastic Pooling**，按照概率大小随机选择窗口内的元素，这也是一种正则化手段。还有**混合池化 Mixed Pooling**，结合最大池化和平均池化，取其加权平均。

**池化层的特性与机制**

- 池化层没有可学习的参数，因为怎么池化只是一个超参数，它只是进行固定的数学运算。池化层通常不需要填充，也不需要 ReLU 激活函数，这属于抗混叠下采样，可以防止下采样过程中的信息混叠。

- 需要注意的是，虽然池化层没有参数，但依然需要反向传播，以便将梯度传回给前面的卷积层。如果使用最大池化，梯度只传给最强激活的位置。如果使用平均池化，梯度均匀分配到窗口内的所有位置。

##### 平移等变性与平移不变性

平移等变性指的是平移和卷积的顺序不重要。数学上表示为：

$$
f(\text{translate}(x)) = \text{translate}(f(x))
$$

这意味着输入图像平移后，输出的特征图也会平移相同的量，但特征值保持不变。这是卷积操作自身的性质，参数共享使得滤波器在平移后依然能捕捉到相同的特征。

而池化层则引入了平移不变性。物体在图像中稍微移动，经过池化下采样后，最大激活值可能依然被保留，最终的分类结果保持不变。两者概念不同，需要区分：卷积是等变的，池化提供了一定的不变性，这共同增强了网络对目标位置变化的鲁棒性。

---





#### 卷积神经网络的训练以及 CNN 架构

##### 怎么构建 CNN

构建一个卷积神经网络，通常是由卷积层、池化层、归一化层、激活函数以及全连接层堆叠而成。卷积层负责提取特征，池化层负责下采样，归一化层负责稳定训练，激活函数引入非线性，全连接层负责最终的分类或回归。

**归一化层**

归一化层的作用是将数据转换为单位高斯分布，通过缩放和平移操作，让每一层的输入分布保持稳定，从而加速训练。主要的区别在于**怎么计算均值和标准差**。

在卷积神经网络中，输入通常是四维张量 $N, C, H, W$。不同的归一化方法在不同的维度上计算统计量：

- **批量归一化 Batch Normalization**：在批次维度 $N$ 和空间维度 $H, W$ 上计算均值和方差，对每个通道 $C$ 独立进行。它要求批次大小不能太小，否则统计量不准确。
- **层归一化 Layer Normalization**：在通道维度 $C$ 和空间维度 $H, W$ 上计算，对每个样本 $N$ 独立进行。它不依赖于批次大小，常用于循环神经网络和 Transformer。
- **实例归一化 Instance Normalization**：只在空间维度 $H, W$ 上计算，对每个样本的每个通道独立进行。常用于风格迁移任务。

![归一化层](../img/Normalization_Layers.png)

**Dropout**

Dropout 是一种在训练时引入随机性的正则化手段。它通过设定一个固定的超参数，即丢弃概率，在每次前向传播时随机将一部分神经元的输出置为 0。

这可以使用掩码技巧来实现，被丢弃的部分不参与本轮的前向和反向计算。

直觉上，这能迫使网络不依赖特定的神经元，从而学习到更泛化的特征。



```python
# Vanilla Dropout: Not recommended implementation (see notes below)
p = 0.5 # probability of keeping a unit active. higher = less dropout

def train_step(X):
    # forward pass for example 3-layer neural network
    H1 = np.maximum(0, np.dot(W1, X) + b1)
    U1 = np.random.rand(*H1.shape) < p # first dropout mask
    H1 *= U1 # drop!
    H2 = np.maximum(0, np.dot(W2, H1) + b2)
    U2 = np.random.rand(*H2.shape) < p # second dropout mask
    H2 *= U2 # drop!
    out = np.dot(W3, H2) + b3

def predict(X):
    # ensembled forward pass
    H1 = np.maximum(0, np.dot(W1, X) + b1) * p # NOTE: scale the activations
    H2 = np.maximum(0, np.dot(W2, H1) + b2) * p # NOTE: scale the activations
    out = np.dot(W3, H2) + b3
```



需要注意的是，测试时不再丢弃任何值。因为训练时按概率 $p$ 随机丢弃，测试时全部保留，这会导致测试时的输入量级比训练时大。为了保证输入量级一致，需要在测试时给激活值乘上 $p$，也就是上面代码中 `* p` 的操作。

（注：现代深度学习框架如 PyTorch 采用的是反向 Dropout。在训练时就将保留的神经元除以 $p$，测试时不做任何处理。这样可以在测试时保持原有前向传播逻辑不变，且只需在训练时缩放一次。）

**激活函数的选择**

早期常用 Sigmoid 作为激活函数。但 Sigmoid 在极负极正区域梯度趋近于 0，随着层数增多，反向传播的梯度会越来越小，产生梯度消失问题，因此 Sigmoid 不再被经常使用。

现在更多使用 ReLU。ReLU 收敛快，但仍有问题：任何负输入都会导致输出为 0，可能产生死神经元。

为了改进，引入了 GELU。GELU 是 Transformer 里主要使用的激活函数，在极端情况下也会逼近 ReLU，但其在零点附近是平滑的，使得优化过程更加稳定。

无论选择哪种，激活函数总是用在线性层（卷积层或全连接层）之后。

##### CNN 的组合架构

**VGGNet**

VGGNet 探索了网络深度对性能的影响。它有一个重要的发现：为什么 3x3 卷积层（步长为 1）有效？

因为堆叠三个 3x3 卷积层的感受野等同于一个 7x7 卷积层，但三个 3x3 层的参数量更少（$3 \times (3^2 C^2) = 27C^2$ 对比 $7^2 C^2 = 49C^2$），且由于中间加入了非线性激活函数，使其建模能力更加复杂，能提取更抽象的特征。

**ResNets**

ResNets 试图解决一个核心问题：叠加更深的层会怎么样？

理论上，深层网络能囊括浅层网络的所有模型，但实验发现，深层网络更难优化，因此可能表现更差。这被称为退化问题，深层网络难以逼近浅层模型的水平，单靠时间无法达到。

ResNets 引入了残差连接。它让网络拟合残差映射，即 $F(x) + x$。当恒等映射是最优解时，网络只需要将残差 $F(x)$ 的权重学习为 0，就能轻松实现恒等映射。直觉上，加入残差连接相当于给梯度提供了一条高速公路，使得深层网络能够更容易地学习恒等映射，从而在极深的网络中也能保持优秀的性能。

##### 权重初始化

**Kaiming 初始化**

权重初始化对训练至关重要。如果权重太小，激活值会迅速衰减到 0；如果权重太大，激活值会迅速爆炸。为了保持每一层激活值的方差稳定，Kaiming 初始化引入了 ReLU 修正。

```python
dims = [4096] * 7
hs = []
x = np.random.randn(16, dims[0])
for Din, Dout in zip(dims[:-1], dims[1:]):
    W = np.random.randn(Din, Dout) * np.sqrt(2/Din) # ReLU correction
    x = np.maximum(0, x.dot(W))
    hs.append(x)
```

通过乘以 $\sqrt{2 / D_{in}}$，激活值的分布能被很好地缩放。在实验中，使用 Kaiming 初始化后，各层激活值的均值和标准差能保持稳定，实现了均值和标准差不随层数加深而剧烈变化的理想效果。

![Kaiming 初始化](../img/Kaiming_Init.png)

对比实验：
- 权重值过小：`W = 0.01 * np.random.randn(...)`，深层网络的激活值趋近于 0。
- 权重值过大：`W = 0.05 * np.random.randn(...)`，激活值迅速爆炸。

![WI1](../img/Weight_Init1.png)
![WI2](../img/Weight_Init2.png)
- 归一化层也能在某种程度上解决激活值爆炸的问题。

##### 怎么训练 CNN

**数据处理与数据增强**

训练 CNN 的第一步是数据处理。通常需要对图像进行归一化，计算整个数据集的均值和标准差，比如使用 ImageNet 的统计量。

数据增强是防止过拟合的重要手段。可以做哪些增强？
- 翻转、缩放、裁剪。需要让人类依然能辨认，但模型更难死记硬背。
- 测试时增强：对同一张图做多种增强，平均预测结果。
- 色彩抖动、亮度变化。
- Cutout（随机裁剪区域置 0）。

**迁移学习 Transfer Learning**

在实际应用中，我们往往没那么多数据。此时可以使用迁移学习。

![迁移学习](../img/Transfer_CNN.png)

如果数据量较小，与 ImageNet 数据相似：换掉 ImageNet 的最后一层全连接层（输出类别数改为自己的类别数），冻结其他层。这相当于把预训练模型当作固定的特征提取器，只训练最后的分类器。

如果数据量较大，可以微调所有层：使用预训练模型初始化，然后微调整个网络。数据越大，就越有利于训练更多的层。

**超参数选择**

超参数选择有一套实用的流程：
1. 小样本过拟合测试，看损失能否快速降低。这通常用于检查代码是否跑通，以及模型是否有能力拟合数据。
2. 粗略的超参数网格，看损失和准确率曲线。
3. 直到训练准确率开始偏离验证准确率，这表明开始过拟合。
4. 过程可以反复进行。
5. 在超参数空间里随机搜索，比如随机选择学习率、正则化强度等。

（注：随机搜索通常比网格搜索更高效，因为并非所有超参数都同等重要。）


---



超参数选择没有绝对完美的固定公式，它高度依赖于具体任务、网络架构和数据集。一般通过反复的实验迭代来寻找最优组合。整套流程可以按照以下阶段逐步展开，并配合监控指标进行动态调整。

**代码正确性验证**

在开始正式的调参之前，一般需要先验证代码逻辑是否无误。

- 若在小样本数据上，训练损失无法快速降低，甚至无法让模型过拟合，则说明代码本身很可能存在缺陷。需要检查数据加载、网络连接、损失函数或者学习率是否过大导致梯度爆炸。
- 若一切正常，训练损失能顺利下降，则进入下一阶段。

**观察曲线变化并对应调整**

在确定了代码没问题之后，通常先使用少量 Epoch 进行粗略的超参数网格搜索，通过观察训练损失和验证准确率曲线的变化趋势，进行针对性调整。这是调参最核心的环节。

- 若训练损失和验证损失都居高不下，模型一般处于欠拟合状态。此时需要增大模型容量，比如增加层数或滤波器数量，或者减少正则化强度，并检查学习率是否过小。
- 若训练损失持续下降，而验证损失开始上升，模型一般进入过拟合状态。此时需要增加数据增强，增大正则化强度，如权重衰减或 Dropout 概率，或者使用早停策略。
- 一旦训练准确率开始偏离验证准确率，且曲线出现明显分叉，一般标志着过拟合开始发生。此时需要及时记录当前的最佳模型权重，并回退到分叉前的超参数设置。
- 若损失曲线出现剧烈震荡甚至发散，一般说明学习率过大。此时需要立即减小学习率，比如将其缩小为原来的十分之一再尝试。

**搜索策略**

在确定了合理的超参数搜索范围后，一般推荐在超参数空间里进行随机搜索。

- 网格搜索会均匀地在网格上取点，但实际上某些超参数对结果的影响远大于另一些。随机搜索能在不考虑网格布局的情况下，更密集地探索对结果影响更大的那个维度，从而在相同的计算预算下找到更优的解。
- 一般通过比较不同超参数组合的最终验证集指标来进行筛选。

**超参数调整的优先级**

超参数的调整本身也有优先级。按照重要性从高到低，依次为：

- 学习率：一般是最重要的超参数。通常需要观察验证集的收敛速度与最终精度。一般建议在 log 空间上进行搜索，例如从 0.1 到 0.0001 之间进行对数均匀采样。
- 正则化强度：一般包括权重衰减系数和 Dropout 概率。通常需参考训练集和验证集准确率之间的差距。若差距过大，一般需增大正则化强度；若整体分数偏低，则可适当减小。
- 批量大小：通常与学习率存在线性缩放关系。当批量大小翻倍时，学习率一般也应当适当调大，以保证梯度更新的方差大致恒定。
- 网络架构超参数：比如层数、滤波器数量和卷积核尺寸。一般在前几项调优完成后再进行微调，通常需观察模型容量是否匹配任务复杂度。

**早停策略**

早停是在实践中非常有效的策略。在验证集准确率连续多个 Epoch 不再提升时，一般应立即停止训练，并保存验证集上表现最好的那组权重。这不仅能节省训练时间，还能防止模型在训练集上过度记忆。

---

序列建模


RNN
隐藏状态
![RNN](../img/RNN.png)

递推公式
ht = fw(ht-1,xt)
yt = fwhy(ht)
把隐藏状态转成输出维度，同时是权重矩阵。

处理向量序列时，每个时间步都套用ht=这个公式（算隐藏状态时），但预测输出时公式不同

vanilla 用 tan当激活函数
公式ht = 

many to many task

隐藏状态需要存什么信息？（上一步的状态，当前值）

![](../img/RNN_exa.png)




#### 序列建模与循环神经网络 RNN

##### 序列建模 Sequence Modeling

在处理图像分类任务时，模型通常是对单张图片进行独立预测。但在许多任务中，数据之间存在时间或顺序上的关联，比如自然语言文本、音频信号、视频帧序列。这些数据通常被称为序列数据，而序列建模的任务就是让模型能够处理这类拥有前后依赖关系的数据。

通常，序列建模需要模型具备“记忆”能力，能够根据之前时刻的信息来影响当前时刻的输出。循环神经网络正是为了解决这类问题而提出的经典架构。

##### RNN 的核心机制

![RNN](../img/RNN.png)

RNN 通过引入隐藏状态来保存历史信息。隐藏状态就像是网络的“记忆”，它会随着时间步的推进而更新，将过去的信息传递到未来。

RNN 的展开结构如上图所示。输入序列为 $x_1, x_2, ..., x_t$，网络在每个时间步接收一个输入，并结合上一时间步的隐藏状态，计算当前的隐藏状态和输出。这种结构被称为展开的 RNN。

RNN 的递推公式如下：

$$
h_t = f_W(h_{t-1}, x_t)
$$

$$
y_t = f_y(W_{hy} h_t)
$$

其中 $h_t$ 是当前时间步的隐藏状态，$h_{t-1}$ 是上一时间步的隐藏状态，$x_t$ 是当前时间步的输入。$y_t$ 是当前时间步的输出。$f_W$ 通常是一个带有非线性激活函数的线性变换，用于更新隐藏状态。$f_y$ 用于把隐藏状态转成输出维度，同时也是一个权重矩阵，负责将隐藏状态映射到输出空间。

在处理向量序列时，每个时间步都会套用 $h_t = f_W(h_{t-1}, x_t)$ 这个公式来计算隐藏状态，但在预测输出时，公式往往与计算隐藏状态的公式不同。

在经典的 vanilla RNN 中，$f_W$ 通常使用 tanh 作为激活函数，以防止梯度在反向传播过程中过快爆炸或消失。其展开形式为：

$$
h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t)
$$

##### Many to Many 任务

RNN 可以用于多种类型的任务，包括 many to one、one to many 以及 many to many。其中 many to many 任务要求模型在每一个时间步都产生一个输出，例如视频的逐帧标注或词性标注。

![RNN实例](../img/RNN_exa.png)

在上图所示的例子中，任务被定义为一个 many to many 的序列建模问题，目标是从输入的序列中检测连续的 1。例如，输入序列为 $0, 1, 0, 1, 1, 1, 0, 1, 1$，模型需要输出对应位置的预测，判断当前或之前是否出现了连续的两个 1。

---


In [ ]:
import numpy as np

# 定义 ReLU 激活函数
def relu(x):
    return np.maximum(0, x)

# 初始化权重矩阵，数值严格按照图中给定
# w_xh: 输入到隐藏状态的权重，形状 (3, 1)
w_xh = np.array([[1], 
                 [0], 
                 [0]])

# w_hh: 隐藏状态到隐藏状态的循环权重，形状 (3, 3)
# 作用是将上一时刻的“当前值”转移到“前一个值”位置，保留常数项 1
w_hh = np.array([[0, 0, 0], 
                 [1, 0, 0], 
                 [0, 0, 1]])

# w_hy: 隐藏状态到输出的权重，形状 (1, 3)
# 权重设置为 [1, 1, -1]，用于计算当前值 + 上一个值 - 1
w_hy = np.array([[1, 1, -1]])

# 输入序列 X，目标是检测连续的 1
x_seq = [0, 1, 0, 1, 1, 1, 0, 1, 1]

# 初始隐藏状态 h_0，形状 (3, 1)
# 包含三个分量：[当前值, 前一个值, 常数项1]
h_t_prev = np.array([[0], 
                     [0], 
                     [1]])

print("X\tRNN\tY")
print("-" * 20)

# 记录每个时间步的输出结果
y_seq = []

for t, x in enumerate(x_seq):
    # 1. 计算当前隐藏状态
    # h_t = ReLU(W_hh * h_{t-1} + W_xh * x_t)
    # 注意：这里的 @ 表示矩阵乘法
    h_t = relu(w_hh @ h_t_prev + w_xh * x)
    
    # 2. 计算当前输出
    # y_t = ReLU(W_hy * h_t)
    y_t = relu(w_hy @ h_t)
    
    # 3. 更新上一时刻的隐藏状态，用于下一次迭代
    h_t_prev = h_t
    
    # 收集输出结果
    y_seq.append(y_t.item())
    
    # 打印当前的输入和输出，展示对应关系
    print(f"{x}\t->\t{int(y_t.item())}")

print("-" * 20)
print("最终预测输出 Y:", y_seq)


##### Vanilla RNN 实例

在具体的代码实现中，这个检测连续 1 的任务被巧妙地通过矩阵运算实现了。代码中定义了三组权重矩阵，它们各自承担了非常明确的功能。

```python
w_xh = np.array([[1], [0], [0]])
w_hh = np.array([[0, 0, 0], [1, 0, 0], [0, 0, 1]])
w_hy = np.array([[1, 1, -1]])
```

**隐藏状态 $h_t$ 的三个分量**

在具体解析之前，需要先明确隐藏状态 $h_t$ 的三个分量所代表的物理意义。根据图中的注释，隐藏状态 $h_t$ 是一个三维向量，其三个分量分别代表：

- 分量1：当前输入的值 Current
- 分量2：上一个输入的值 Previous
- 分量3：常数项 1，用于提供偏置 Bias

**权重矩阵的具体工作**

1. **输入权重矩阵 `w_xh`**

   `w_xh` 的形状是 $3 \times 1$。它的作用是把输入标量 $x_t$ 映射到隐藏状态向量的三个维度上。
   根据矩阵乘法，$[3 \times 1] \times x_t = [3 \times 1]$，结果相当于把 $x_t$ 放到第一个分量，第二个和第三个分量置为 0。这正好对应了“将当前值记录到隐藏状态的第一个分量中”这一逻辑。

2. **循环权重矩阵 `w_hh`**

   `w_hh` 的形状是 $3 \times 3$。它的作用是将上一时刻的隐藏状态 $h_{t-1}$ 进行线性组合，形成当前时刻隐藏状态的初始部分。
   通过矩阵乘法，$w_{hh} \times h_{t-1}$ 完成的是：把上一时刻的“当前值”复制到当前时刻的“上一个值”位置，同时保留常数项 1，并重置“当前值”为 0。
   这个操作实现了状态的转移：当前时刻的“历史”变成了上一时刻的“实时”。

3. **输出权重矩阵 `w_hy`**

   `w_hy` 的形状是 $1 \times 3$。它的作用是从隐藏状态中提取信息，计算最终输出。
   它的权重是 $[1, 1, -1]$。结合隐藏状态 $h_t$ 的三个分量，输出计算为：$1 \times \text{Current} + 1 \times \text{Previous} - 1 \times 1$。
   具体逻辑是：当且仅当“当前值”和“上一个值”都为 1 时，计算结果为 $1+1-1=1$。否则，如果只有一个 1，结果为 $1+0-1=0$；如果没有 1，结果为 $0+0-1=-1$。

**前向传播与激活函数**

```python
h_t_prev = np.array([[0], [0], [1]])
for t, x in enumerate(x_seq):
    h_t = relu(w_hh @ h_t_prev + w_xh @ x)
    y_t = relu(w_hy @ h_t)
    h_t_prev = h_t
```

循环中的代码使用了 ReLU 作为激活函数。对于 $h_t$，由于 $w_{hh} @ h_{t-1} + w_{xh} @ x_t$ 产生的结果中，前两个分量通常大于等于 0，而第三个分量恒为 1，ReLU 实际上起到了一个取最大值的作用。在这种情况下，$h_t$ 的三个分量始终保持非负。

对于 $y_t$，由于 $w_{hy} @ h_t$ 计算结果可能为 $-1$、$0$ 或 $1$。当输出为 $1$ 时，经过 ReLU 后依然为 $1$；当输出为 $0$ 时，经过 ReLU 后依然为 $0$；当输出为 $-1$ 时，经过 ReLU 后变成 $0$。

因此，这个具体的 RNN 例子实际上是通过手工设计权重，实现了在任意序列上检测是否出现了连续两个 1 的功能。这也是 RNN 前向传播的一个很直观的示例：通过矩阵运算和激活函数，让隐藏状态不断更新，从而提取序列中的时序依赖关系。


RNN的loss和梯度的计算。

梯度相加。
有时只需要最后的梯度、有时需要隐藏矩阵的梯度（视频分类）

每个时间步的激活值和梯度都储存下来，序列长开销会很大

随时间反向传播

截断随时间反向传播（分批处理，批次间梯度不再向后传播）
Truncated BPTT


怎么从模型里采样?
不总是选择概率最高的

searching for interpretable cells
行长追踪单元...


什么时候该用RNN，他好在哪

没有上下文长度限制（理论上）

模型大小不会随时间步变成而增加
...
缺点呢？
递归计算，耗时
实际上很难获取很多时间步之前的信息
...
RNN应用
图像描述生成（幻觉，建立了连接没建立解释）
视觉问答
视觉导航任务

MUltilayer RNNs

LSTM
RNN有先天不足，容易梯度消失（或者梯度爆炸）
信息捕获不足

长短期记忆网络。
门控。细胞状态，遗忘门...


#### RNN 的损失与梯度计算

RNN 的训练过程与普通的全连接网络类似，但因为涉及到时间维度，其前向传播和反向传播都必须在序列上逐步展开。

##### 前向传播与损失计算

以 Andrej Karpathy 的极简字符级 RNN 代码 `min-char-rnn.py` 为例，模型在每一个时间步接收一个字符输入，输出下一个字符的预测概率分布。

前向传播的核心流程如下：

1. 计算当前隐藏状态：$h = \tanh(W_{xh}x + W_{hh}h_{prev} + b_h)$
2. 计算未归一化的输出分数：$y = W_{hy}h + b_y$
3. 通过 softmax 得到概率分布：$p = \exp(y) / \sum \exp(y)$
4. 根据目标字符计算交叉熵损失，并累加到总损失中。

**随时间反向传播**

反向传播过程中，误差需要从序列的末端逐级向前传递，这被称为随时间反向传播 Backpropagation Through Time, BPTT。

序列中的每一个时间步都会对最终的损失函数有贡献。因此，损失函数对每个时间步的隐藏状态的梯度都需要累加。
```python
dh = np.dot(Why.T, dy) + dh_next # 结合当前步和未来步的梯度
dhraw = (1 - h * h) * dh          # 经过 tanh 的反向传播
dbh += dhraw                      # 累加偏置梯度
dWxh += np.dot(dhraw, xs[t].T)    # 累加输入权重梯度
dWhh += np.dot(dhraw, hprev.T)    # 累加循环权重梯度
```
在计算 `dh` 时，`dh_next` 是未来时间步传回来的梯度，这一步实现了梯度在时间上的反向流动。

因为参数 $W_{xh}$、$W_{hh}$、$W_{hy}$ 在时间维度上是共享的，所以它们在整个序列上的梯度是各个时间步梯度累加的结果。这就是笔记中提到的“梯度相加”。

根据任务需求，有时只需要最后一个时间步的梯度用于分类，有时则需要每个时间步的隐藏状态梯度（例如视频分类中的逐帧标注）。因此，需要将每个时间步的激活值和梯度都储存下来，但这会导致长序列的内存开销急剧增大。

**截断随时间反向传播**

为了解决长序列内存开销过大的问题，引入了截断随时间反向传播 Truncated BPTT。

它的做法是将长序列切分成若干个固定长度的子序列，比如每 50 个时间步作为一个批次。前向传播和反向传播只在子序列内部进行，批次与批次之间不再传递梯度。这使得计算图的大小被固定，内存开销变得可控，但代价是模型无法捕获超过这个截断长度的长期依赖关系。

**梯度裁剪**

在 BPTT 过程中，梯度可能会因为链式法则的累积而爆炸。因此，通常需要加入梯度裁剪，将梯度的绝对值限制在某个范围内，例如 5 以内。
```python
for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam)
```

这个步骤能够有效防止梯度爆炸，保证训练的稳定性。

**从模型里采样**

在生成文本时，模型并不总是选择概率最高的字符。如果总是选择概率最高的字符，生成的文本会非常单一且缺乏创造性。因此，通常采用随机采样，根据输出的概率分布进行抽样，这样可以在保证合理性的同时，增加生成结果的多样性。

**搜索可解释单元**

训练好的 RNN 模型内部可能存在一些有特定功能的神经元，比如行长追踪单元，它们能够追踪文本中引号的开闭状态或代码块的缩进层级。通过搜索这些可解释单元，可以帮助理解模型究竟在多大程度上学到了符号级的规律。

#### RNN 的优缺点与应用

**优点**

- 理论上，RNN 没有上下文长度的限制，它可以处理任意长度的序列。
- 模型的大小不会随时间步的增加而增大，因为所有权重在时间步上都是共享的。
- 这使得 RNN 非常适合处理自然语言和音频等变长序列。

**缺点**

- RNN 采用递归计算，导致计算过程无法并行化，耗时较长。
- 实际上 RNN 很难获取很多时间步之前的信息，因为梯度在反向传播过程中容易消失或爆炸，导致长距离依赖难以学习。

##### RNN 的应用

在计算机视觉领域，RNN 也有广泛的应用：

- 图像描述生成：模型输入一张图片，输出一段描述该图片的文字。但这种应用存在幻觉问题，模型可能建立了某种连接，但并没有真正建立可解释的因果理解。
- 视觉问答：模型需要根据图像内容回答自然语言问题。
- 视觉导航任务：模型需要根据视觉输入做出连续的路径决策。

#### 多层 RNN 与 LSTM

**多层 RNNs**

多层 RNN 通过在深度方向叠加多个 RNN 层，让第一层的输出作为第二层的输入，以此类推。这增加了模型的容量，使得网络能够学习到更高级的抽象特征。

**LSTM**

RNN 有先天不足，容易发生梯度消失或梯度爆炸，导致信息捕获不足。为了解决这个问题，引入了长短期记忆网络 LSTM。

LSTM 引入了门控机制和细胞状态。其核心机制可以拆解为以下几个部分：

- 细胞状态：相当于一条信息高速公路，梯度可以沿着它几乎无衰减地传播。
- 遗忘门：决定丢弃哪些旧信息。
- 输入门：决定写入哪些新信息。
- 输出门：决定输出哪些信息。

这使得 LSTM 能够在长序列中有效地捕获并保留长期依赖关系，成为 RNN 家族中最具代表性的改进架构。


In [ ]:
import numpy as np

# 1. 数据准备
# 演示用文本数据，直接内置避免依赖外部文件
data = "hello world. this is a simple char rnn example. " * 20
chars = list(set(data)) # 获得所有不重复字符
data_size, vocab_size = len(data), len(chars)
print(f"数据长度: {data_size}, 词汇表大小: {vocab_size}")

# 字符与整数索引的互相映射
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

# 2. 超参数与初始化
hidden_size = 100    # 隐藏层神经元数量
seq_length = 25      # 每个批次的序列长度 (BPTT 的截断长度)
learning_rate = 1e-1 # 学习率

# 模型参数初始化 (使用小方差随机初始化，防止初始饱和)
Wxh = np.random.randn(hidden_size, vocab_size) * 0.01  # 输入到隐藏层的权重
Whh = np.random.randn(hidden_size, hidden_size) * 0.01 # 隐藏层循环权重
Why = np.random.randn(vocab_size, hidden_size) * 0.01  # 隐藏层到输出层的权重
bh = np.zeros((hidden_size, 1))                        # 隐藏层偏置
by = np.zeros((vocab_size, 1))                         # 输出层偏置

# Adagrad 优化器内存变量 (累积梯度平方，实现自适应学习率)
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by)

# 3. 前向与反向传播
def lossFun(inputs, targets, hprev):
    """
    前向计算损失，反向计算梯度 (BPTT)。
    inputs/targets: 字符索引列表
    hprev: 上一批次的最终隐藏状态 (用于批次间传递)
    """
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev) # 初始化前一步隐藏状态
    loss = 0
    
    # ---- 前向传播 ----
    for t in range(len(inputs)):
        xs[t] = np.zeros((vocab_size, 1)) # one-hot 编码
        xs[t][inputs[t]] = 1
        
        # 计算当前隐藏状态：结合当前输入和上一隐藏状态
        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh)
        
        # 计算未归一化得分 logits 和 softmax 概率
        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
        
        # 累加交叉熵损失
        loss += -np.log(ps[t][targets[t], 0])
        
    # ---- 反向传播 (BPTT) ----
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dh_next = np.zeros_like(hs[0]) # 未来时间步传回的梯度
    
    for t in reversed(range(len(inputs))):
        # softmax + 交叉熵的反向传播
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1 # 目标类概率减1，其余不变
        
        dWhy += np.dot(dy, hs[t].T)
        dby += dy
        
        # 梯度相加：当前步输出贡献 + 未来步传回
        dh = np.dot(Why.T, dy) + dh_next
        
        # 反向传播通过 tanh 激活函数
        dhraw = (1 - hs[t] * hs[t]) * dh 
        
        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)
        
        # 将梯度继续向前一时间步传递
        dh_next = np.dot(Whh.T, dhraw)
        
    # 梯度裁剪，防止梯度爆炸
    for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(dparam, -5, 5, out=dparam)
        
    return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

# 4. 采样生成
def sample(h, seed_ix, n):
    """根据当前模型和隐藏状态，生成指定长度的字符序列"""
    x = np.zeros((vocab_size, 1))
    x[seed_ix] = 1 # 种子字符
    ixes = []
    
    for t in range(n):
        h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)
        y = np.dot(Why, h) + by
        p = np.exp(y) / np.sum(np.exp(y))
        
        # 按概率随机采样，而不是每次取最大值
        ix = np.random.choice(range(vocab_size), p=p.ravel()) 
        x = np.zeros((vocab_size, 1))
        x[ix] = 1
        ixes.append(ix)
        
    return ixes

# 5. 主训练循环
n, p = 0, 0
# 平滑损失，用于过滤单次波动，观察整体趋势
smooth_loss = -np.log(1.0 / vocab_size) * seq_length 

while True:
    # 当数据指针走到末尾，或者刚开始时，重置指针和隐藏状态
    if p + seq_length + 1 >= len(data) or n == 0:
        hprev = np.zeros((hidden_size, 1))
        p = 0
        
    # 准备输入和目标序列 (目标向右偏移一位)
    inputs = [char_to_ix[ch] for ch in data[p:p+seq_length]]
    targets = [char_to_ix[ch] for ch in data[p+1:p+seq_length+1]]
    
    # 每 100 步采样一次，观察生成效果
    if n % 100 == 0:
        sample_ix = sample(hprev, inputs[0], 200)
        txt = ''.join(ix_to_char[ix] for ix in sample_ix)
        print(f"----\n {txt} \n----")
        
    # 前向与反向传播
    loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
    smooth_loss = smooth_loss * 0.999 + loss * 0.001
    
    if n % 100 == 0:
        print(f"迭代次数: {n}, 平滑损失: {smooth_loss}")
        
    # Adagrad 参数更新 (自适应调整每个参数的学习率)
    for param, dparam, mem in zip([Wxh, Whh, Why, bh, by],
                                  [dWxh, dWhh, dWhy, dbh, dby],
                                  [mWxh, mWhh, mWhy, mbh, mby]):
        mem += dparam * dparam
        param += -learning_rate * dparam / np.sqrt(mem + 1e-8)
        
    p += seq_length
    n += 1